# SavorLink Strategy III: Tanimoto Pseudo-Edges for Cold-Start Ingredients

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Goal**: Predict food pairings for **cold-start ingredients** (zero recipe co-occurrence edges) in a food-chemical graph by injecting **soft, weighted pseudo-edges** connecting the cold node to its chemically most similar known hub ingredients (weights = Tanimoto similarity on Morgan/ECFP fingerprints).

### Core Design Principles
1. **Strictly Inductive Architecture**: No transductive `nn.Embedding(num_nodes)` lookup table. Cold-start nodes are encoded purely through chemistry (2048-bit Morgan fingerprints) and topological pseudo-edges.
2. **Zero Leakage Protocol**: Test ingredients have **all** real co-occurrence edges removed from the training graph. Node features derive exclusively from chemical structures, never recipe co-occurrence.
3. **Tanimoto-Scaled Message Passing**: Dual-branch GNN (`WeightedSAGEConv` / `WeightedGATConv`) where pseudo-edge messages are normalized and weighted by chemical similarity $w_{uv} \in [0, 1]$.
4. **Google Colab Free-Tier Optimized**: Fits completely inside free Tesla T4 GPU ($<100$ MB VRAM, $<1$ GB system RAM, well below 16 GB limits).
5. **Portable Model Export**: Generates exportable `.pt` weights, `.pkl` checkpoints, and a 1-click download of `savorlink_model_exports.zip` for downstream inference.

In [ ]:
!rm -rf data/cache data/exports
!unzip -o -q colab_bundle.zip


## 1. Hardware Verification & Dependency Installation

In [ ]:
# Verify GPU hardware
import torch
import os
import sys

print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Running in CPU mode. (Tip: In Colab, go to Runtime -> Change runtime type -> T4 GPU)")
print("=" * 60)

# Install dependencies (Colab-compatible)
!pip install rdkit pubchempy pyyaml scipy scikit-learn tqdm --quiet

PyTorch Version: 2.11.0+cu130
CUDA Available:  True
GPU Device:      Tesla T4
Device Memory:   14.56 GB


## 2. Workspace Setup & Dataset Preparation

Choose **Option A** (Google Drive mount) or **Option B** (Direct project upload / git clone).

In [ ]:
# === OPTION A: Google Drive Mount (Recommended if you have the 241MB walk file in Drive) ===
# from google.colab import drive
# drive.mount('/content/drive')
# %cd '/content/drive/MyDrive/jackfruit'

# === OPTION B: Direct Project Directory Setup ===
import os, sys

# Unzip colab_bundle.zip automatically if present in current directory or /content
if os.path.exists("colab_bundle.zip"):
    !unzip -o -q colab_bundle.zip
elif os.path.exists("/content/colab_bundle.zip"):
    !unzip -o -q /content/colab_bundle.zip -d /content

# Ensure project root is on Python module search path
for p in [os.path.abspath("."), "/content"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

!mkdir -p data/raw data/cache data/splits data/exports/checkpoints

# Download official FlavorGraph CSV files if not already present
if not os.path.exists("data/raw/nodes_191120.csv"):
    !curl -sL -o data/raw/nodes_191120.csv "https://raw.githubusercontent.com/lamypark/FlavorGraph/master/input/nodes_191120.csv"
if not os.path.exists("data/raw/edges_191120.csv"):
    !curl -sL -o data/raw/edges_191120.csv "https://raw.githubusercontent.com/lamypark/FlavorGraph/master/input/edges_191120.csv"

print("Project structure verified:")
!ls -lh data/raw

Project structure verified:
total 5.3M
-rw-r--r-- 1 root root 5.0M Nov  3 08:26 edges_191120.csv
-rw-r--r-- 1 root root 336K Nov  3 08:26 nodes_191120.csv


### (Optional) Metapath Walk Corpus Integration
If you have the metapath random-walk corpus file (`M17-metapath_CHNHC+NHCHN+RandWalk-whichmeta_100-num_walks_50-len_metapath.txt`), you can upload it or mount Google Drive.
If not present, the pipeline automatically falls back to FlavorGraph's 111k CSV NPMI edges.

In [ ]:
walk_file = "M17-metapath_CHNHC+NHCHN+RandWalk-whichmeta_100-num_walks_50-len_metapath.txt"
if os.path.exists(walk_file):
    print(f"Found Metapath walk corpus: {walk_file} ({os.path.getsize(walk_file) / (1024*1024):.1f} MB)")
else:
    print("Metapath walk file not found locally. Using FlavorGraph CSV NPMI edges.")
    # To mount Google Drive if file is stored there, uncomment:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # !cp '/content/drive/MyDrive/path/to/M17-metapath...txt' .

Found Metapath walk corpus: M17-metapath_CHNHC+NHCHN+RandWalk-whichmeta_100-num_walks_50-len_metapath.txt (241.0 MB)


## 3. Data Loading & Resumable PubChem SMILES Cache

In [ ]:
import os, sys
for p in [os.path.abspath("."), "/content"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

import yaml
from src.data.loaders import load_config, load_flavorgraph, print_data_report
from src.data.pubchem_lookup import PubChemCache, build_smiles_cache, coverage_report

cfg = load_config("configs/default.yaml")
if not os.path.exists(cfg.get("paths", {}).get("walk_corpus", "")):
    cfg["paths"]["walk_corpus"] = None

fg = load_flavorgraph(cfg, project_root=".")
print_data_report(fg)

# Populate PubChem SMILES cache (resumable with disk caching)
cache = PubChemCache(cfg["paths"]["pubchem_cache"])
cids = {c.pubchem_cid for c in fg.compounds.values() if c.pubchem_cid is not None}
build_smiles_cache(cids, cache, cfg["pubchem"])
print(coverage_report(fg, cache))

FlavorGraph Data Report

Total nodes: 8298
  Ingredients: 6653
    Hub ingredients: 416
  Compounds: 1645
    Food compounds: 1561
    Drug compounds: 84
    With PubChem CID: 1645/1645

Total edges: 147179
  ingr-ingr: 111355
  ingr-fcomp: 35440
  ingr-dcomp: 384

Ingredients with compounds: 417/6653
  Compounds per ingredient: mean=85.9, max=408, median=94

Ingr-ingr NPMI scores (111355 edges):
  Range: [0.0000, 0.8582]
  Mean: 0.1972
  Median: 0.1873
  Negative: 0

Ingr-ingr degree distribution:
  Mean: 33.5, Max: 1080
  Zero-degree ingredients: 0
PubChem SMILES Coverage Report

Compound-level:
  Total compounds: 1645
  With PubChem CID: 1645 (100.0%)
  With valid SMILES in cache: 1645 (100.0%)

Ingredient-level:
  Total ingredients: 6653
  With any compound links: 417
  With valid SMILES in cache: 417
  Without compound links (recipe-only): 6236
  Valid SMILES per ingredient (where >0): mean=85.9, max=408


## 4. Chemical Fingerprints & Sparse Top-$k$ Similarity

In [ ]:
import os, sys
for p in [os.path.abspath("."), "/content"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from src.chem.fingerprints import compute_compound_fingerprints, pool_mean
from src.chem.similarity import compute_sparse_topk_similarity
from src.graph.build_graph import assemble_feature_matrix, build_node_id_mapping
import time

# 1. Compute 2048-bit Morgan Fingerprints
cid_to_smiles = {cid: cache.get(cid) for cid in cids if cache.get(cid) is not None}
cid_to_fp = compute_compound_fingerprints(cid_to_smiles, n_bits=2048)

# 2. Pool compound fingerprints per ingredient
node_id_to_cid = {nid: node.pubchem_cid for nid, node in fg.compounds.items() if node.pubchem_cid is not None}
pooled_fps = {}
for ingr_id, comp_ids in fg.ingredient_compounds.items():
    fps = [cid_to_fp[node_id_to_cid[cid]] for cid in comp_ids if cid in node_id_to_cid and node_id_to_cid[cid] in cid_to_fp]
    if fps:
        pooled_fps[ingr_id] = pool_mean(fps)

all_ingr_ids = sorted(fg.ingredients.keys())
raw_to_idx, idx_to_raw = build_node_id_mapping(all_ingr_ids)
x_matrix = assemble_feature_matrix(all_ingr_ids, raw_to_idx, pooled_fps, n_bits=2048)

# 3. Compute sparse top-k chemical similarities against 416 hub ingredients
hub_ids = sorted(fg.hub_ingredients)
t0 = time.time()
topk_similarities = compute_sparse_topk_similarity(
    query_ids=all_ingr_ids,
    candidate_ids=hub_ids,
    ingredient_compounds=fg.ingredient_compounds,
    node_id_to_cid=node_id_to_cid,
    cid_to_fp=cid_to_fp,
    strategy=cfg["similarity"].get("strategy", "mean_best_match"),
    top_k=cfg["similarity"].get("top_k", 10),
    threshold=cfg["similarity"].get("threshold", 0.05),
    pooled_fps=pooled_fps,
    cache_path="data/cache/topk_similarities.pkl",
)
print(f"Computed sparse top-10 similarities for {len(all_ingr_ids)} ingredients in {time.time() - t0:.2f} s")

Computed sparse top-10 similarities for 6653 ingredients in 155.99 s


## 5. Strict Zero-Leakage Ingredient Splitting

Simulates cold-start ingredients: Test ingredients have **all** edges removed from the graph. Ground truth pairs are held out for ranking evaluation.

In [ ]:
from src.data.splits import create_ingredient_split, verify_no_leakage

split = create_ingredient_split(
    fg,
    test_frac=cfg["splits"].get("test_frac", 0.15),
    val_frac=cfg["splits"].get("val_frac", 0.10),
    min_degree=cfg["splits"].get("min_degree", 5),
    hub_only=cfg["splits"].get("hub_only", True),
    seed=42,
)
violations = verify_no_leakage(split, fg)
assert len(violations) == 0, f"Leakage detected: {violations}"

print(f"Train Ingredients: {len(split.train_ids):,}")
print(f"Validation Ingredients: {len(split.val_ids):,}")
print(f"Test Ingredients (Simulated Cold-Start): {len(split.test_ids):,}")
print(f"Train Graph Edges: {len(split.train_edges):,}")
print(f"Held-Out Test Ground Truth Pairs: {len(split.test_ground_truth):,}")

Train Ingredients: 6,556
Validation Ingredients: 39
Test Ingredients (Simulated Cold-Start): 58
Train Graph Edges: 103,563
Held-Out Test Ground Truth Pairs: 4,699


## 6. Cold-Start Aware Inductive Training

We train `Strategy3Model` with per-epoch cold-masking: randomly chosen training ingredients have all real edges masked, and Tanimoto pseudo-edges injected so the GNN learns to aggregate over chemistry.

In [ ]:
from src.train import train_single_seed, export_artifacts

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on device: {device}")

# Set epochs for Colab run (default: 50 for quick convergence, or 200 for full run)
train_cfg = dict(cfg)
train_cfg["training"] = dict(cfg.get("training", {}))
train_cfg["training"]["epochs"] = 50
train_cfg["training"]["patience"] = 15

summary_metrics, trained_model = train_single_seed(
    fg=fg,
    split=split,
    x_matrix=x_matrix,
    topk_similarities=topk_similarities,
    raw_to_idx=raw_to_idx,
    config=train_cfg,
    seed=42,
    device=device,

)

Training on device: cuda


## 7. Comprehensive Multi-Baseline Evaluation

We compare Strategy III against four standard baselines:
1. **Random**: Chance ranking floor
2. **Popularity**: Training graph degree ranking
3. **Tanimoto-kNN (NO GNN)**: Weighted average of chemical neighbors' co-occurrences
4. **Zero-degree GNN**: Chemistry features only, NO pseudo-edges (failure mode)
5. **Strategy III (Full)**: Tanimoto-weighted message passing GNN

In [ ]:
from src.evaluate import format_metrics_table

print("=" * 68)
print("TEST SET RANKING BENCHMARK (Simulated Cold-Start Ingredients)")
print("=" * 68)
print(format_metrics_table(summary_metrics))
print("=" * 68)

TEST SET RANKING BENCHMARK (Simulated Cold-Start Ingredients)
| Model / Baseline | Recall@10 | Recall@20 | MRR | NDCG@20 | Spearman |
| --- | --- | --- | --- | --- | --- |
| **Random** | 0.0010 ± 0.0046 | 0.0019 ± 0.0072 | 0.0408 ± 0.1408 | 0.0061 ± 0.0242 | -0.0111 ± 0.1740 |
| **Popularity** | 0.0227 ± 0.0397 | 0.0454 ± 0.0764 | 0.2040 ± 0.2972 | 0.0525 ± 0.0801 | -0.0918 ± 0.3701 |
| **Tanimoto-kNN** | 0.0177 ± 0.0320 | 0.0256 ± 0.0381 | 0.1769 ± 0.2516 | 0.0515 ± 0.0771 | -0.0243 ± 0.2146 |
| **Zero-degree GNN** | 0.0005 ± 0.0024 | 0.0030 ± 0.0079 | 0.0467 ± 0.1304 | 0.0063 ± 0.0164 | 0.0105 ± 0.2540 |
| **Strategy III** | 0.0010 ± 0.0047 | 0.0031 ± 0.0078 | 0.0557 ± 0.1379 | 0.0073 ± 0.0159 | 0.0139 ± 0.2575 |


## 8. Export Model & 1-Click Download

All artifacts (PyTorch weights `.pt`, pickle checkpoint `.pkl`, graph metadata, top-k chemical neighbors, test splits, feature matrix `.npy`, and adjacency matrix `.npz`) are bundled into `data/savorlink_model_exports.zip` and downloaded to your machine.

In [ ]:
# Export artifacts to data/exports and create zip bundle
from src.train import export_artifacts, create_export_zip

export_artifacts(
    model=trained_model,
    fg=fg,
    split=split,
    topk_similarities=topk_similarities,
    raw_to_idx=raw_to_idx,
    idx_to_raw=idx_to_raw,
    config=train_cfg,
    seed=42,
    export_dir="data/exports",
    x_matrix=x_matrix,
)

archive_path = create_export_zip("data/exports")

print(f"Portable Export Bundle Created: {archive_path}")
print(f"File Size: {os.path.getsize(archive_path) / (1024*1024):.2f} MB")

# Trigger 1-click browser download in Google Colab
try:
    from google.colab import files
    print("Triggering browser download...")
    files.download(str(archive_path))
except ImportError:
    print(f"Running locally. Export archive is ready at: {os.path.abspath(archive_path)}")

Portable Export Bundle Created: data/savorlink_model_exports.zip
File Size: 5.08 MB
Triggering browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Interactive Food Pairing Inference

Load the self-contained inference bundle directly and query pairings for any ingredient (including held-out cold-start items and custom chemical formulas).

In [ ]:
from src.inference import load_inference_bundle
import random

# 1. Load predictor directly from exported zip archive
predictor = load_inference_bundle("savorlink_model_exports.zip", seed=42, device="cpu")

# 2. Pick a cold-start test ingredient (zero training edges)
test_cold_id = random.choice(split.test_ids)
cold_ingredient_name = fg.ingredients[test_cold_id].name

print(f"\n>>> Query: Cold-Start Ingredient '{cold_ingredient_name}' (ID: {test_cold_id}) <<<")
recommendations = predictor.recommend_pairings(test_cold_id, top_k=10)

# 3. Print ranked pairing predictions
print(f"{'Rank':<5} {'Ingredient':<30} {'Predicted Score':<15}")
print("-" * 52)
for r in recommendations:
    print(f"{r['rank']:<5} {r['name']:<30} {r['score']:<15.4f}")

# 4. Compare with known ground truth recipe partners
true_partners = [fg.ingredients[d].name for (s, d) in split.test_ground_truth if s == test_cold_id or d == test_cold_id]
print(f"\nKnown Ground Truth Recipe Partners ({len(true_partners)} total):")
print(", ".join(true_partners[:12]) if true_partners else "None in training graph (isolated)")


>>> Query: Cold-Start Ingredient 'raisin_bread' (ID: 5192) <<<
Rank  Ingredient                     Predicted Score
----------------------------------------------------
1     bluefish                       0.1664         
2     hot_dog                        0.1664         
3     condensed_milk                 0.1664         
4     fruit_salad                    0.1664         
5     albacore_tuna                  0.1664         
6     cuttlefish                     0.1664         
7     smelt                          0.1616         
8     squirrel                       0.1605         
9     sweet_marjoram                 0.1600         
10    ginkgo_nut                     0.1590         

Known Ground Truth Recipe Partners (27 total):
raisin_bread, raisin_bread, raisin_bread, raisin_bread, raisin_bread, walnut, raisin_bread, raisin_bread, raisin_bread, raisin_bread, raisin_bread, vanilla


In [ ]:
# Example 2: Predict food pairings for a custom novel ingredient
# Defined purely by compound PubChem CIDs (e.g. Eugenol CID 3314, Vanillin CID 1183, Cinnamaldehyde CID 637511)
custom_cids = [3314, 1183, 637511]
print("\n>>> Query: Custom Novel Ingredient with CIDs: [3314 (Eugenol), 1183 (Vanillin), 637511 (Cinnamaldehyde)] <<<")

custom_recommendations = predictor.predict_new_compounds(custom_cids, top_n=10)
print(f"{'Rank':<5} {'Recommended Partner':<30} {'Affinity Score':<15}")
print("-" * 52)
for r in custom_recommendations:
    print(f"{r['rank']:<5} {r['name']:<30} {r['score']:<15.4f}")